# Task 2 — Train Two Supervised Models
Two supervised classification models are trained using the same preprocessing pipeline:
1. Logistic Regression
2. Decision Tree Classifier
Both models are fitted only on the training data. The hold-out test set is not used during training.

In [7]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Load dataset
adult = fetch_openml("adult", version=2, as_frame=True)
df = adult.frame.copy()
df = df.replace("?", np.nan)
df["target"] = df["class"].map({"<=50K": 0, ">50K": 1})
X = df.drop(columns=["class", "target"])
y = df["target"]
numeric_features = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week"
]
categorical_features = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country"
]
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)

# Recreate the same split used in Task 1
X_train_dev, X_test, y_train_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_dev,
    y_train_dev,
    test_size=0.125,
    random_state=42,
    stratify=y_train_dev
)

# Logistic Regression Pipeline
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                solver="liblinear",
                penalty="l2",
                random_state=42,
                max_iter=1000
            )
        )
    ]
)

# Decision Tree Pipeline
decision_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)

# Train Logistic Regression
print("Training Logistic Regression...")
logistic_pipeline.fit(X_train, y_train)
print("Logistic Regression training completed!")

# Train Decision Tree
print("\nTraining Decision Tree...")
decision_tree_pipeline.fit(X_train, y_train)
print("Decision Tree training completed!")

# Generate predictions on development set
logistic_dev_predictions = logistic_pipeline.predict(X_dev)
tree_dev_predictions = decision_tree_pipeline.predict(X_dev)

# Generate development probabilities
logistic_dev_probabilities = logistic_pipeline.predict_proba(X_dev)[:, 1]
tree_dev_probabilities = decision_tree_pipeline.predict_proba(X_dev)[:, 1]

# Training scores
logistic_train_score = logistic_pipeline.score(X_train, y_train)
tree_train_score = decision_tree_pipeline.score(X_train, y_train)

# Development scores
logistic_dev_score = logistic_pipeline.score(X_dev, y_dev)
tree_dev_score = decision_tree_pipeline.score(X_dev, y_dev)

# Display results
print("MODEL TRAINING SUMMARY")
print("\nLogistic Regression")
print("Training Accuracy:", round(logistic_train_score, 4))
print("Development Accuracy:", round(logistic_dev_score, 4))
print("\nDecision Tree")

print("Training Accuracy:", round(tree_train_score, 4))
print("Development Accuracy:", round(tree_dev_score, 4))

# Verify that the hold-out test set was not used
print("DATA USAGE CHECK")
print("Models were fitted using TRAINING data only.")
print("Development data was used only for development evaluation.")
print("Hold-out TEST data was NOT used during training.")


Training Logistic Regression...


c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Logistic Regression training completed!

Training Decision Tree...
Decision Tree training completed!
MODEL TRAINING SUMMARY

Logistic Regression
Training Accuracy: 0.8515
Development Accuracy: 0.853

Decision Tree
Training Accuracy: 0.9999
Development Accuracy: 0.8151
DATA USAGE CHECK
Models were fitted using TRAINING data only.
Development data was used only for development evaluation.
Hold-out TEST data was NOT used during training.


# Task 2 — Conclusion
Two supervised learning models, Logistic Regression and Decision Tree Classifier, were successfully implemented using Scikit-learn pipelines. Both models include the preprocessing pipeline to ensure that missing values, numerical features, and categorical features are handled consistently.
Both models were trained using the training dataset only. The hold-out test set was not used during training, which prevents data leakage and keeps it available for unbiased evaluation in Task 3.
